# 🖼️ Multimodal RAG with GPT-4o-Family Models

Many documents contain a mixture of content types, including text, tables and images.

Yet, information captured in images is lost in most RAG applications.

With the emergence of multimodal LLMs, like [GPT-4o](https://openai.com/index/hello-gpt-4o/), it is worth considering how to utilize images in RAG Systems:

![](https://i.imgur.com/wcCDT38.gif)


---

* Use a multimodal LLM (such as [GPT-4o](https://openai.com/index/hello-gpt-4o/), [LLaVA](https://llava.hliu.cc/), or [FUYU-8b](https://www.adept.ai/blog/fuyu-8b)) to produce text summaries from images and tables
* Embed and retrieve image, table and text summaries with a reference to the raw images and tables for given queries
* Pass raw images, tables and text chunks to a multimodal LLM for answer synthesis   

---



* We will use [Unstructured](https://unstructured.io/) to parse images, text, and tables from documents (PDFs).
* We will use the [multi-vector retriever](https://python.langchain.com/docs/modules/data_connection/retrievers/multi_vector) with [Chroma](https://www.trychroma.com/) and Redis to store raw text and images along with their summaries for retrieval.
* We will use `gpt-4o-mini`, the smaller multimodal model of the GPT-4o family, for text, table and image summarization (for retrieval) as well as for final answer synthesis from a joint review of images and texts (or tables). Pass `model="gpt-4o"` to the factory for the larger model.


___Created By: [Dipanjan (DJ)](https://www.linkedin.com/in/dipanjans/)___

## Learning Objectives
In this notebook, you will learn:
1. **Partition a PDF** - extract text chunks, tables and images with Unstructured
2. **Prepare tables and images** - convert HTML tables to Markdown and inspect the extracted figures
3. **Summarize every element** - text, table and image summaries from a multimodal LLM, optimized for retrieval
4. **Multi-vector retriever** - index the summaries in Chroma and return the raw elements from a Redis docstore
5. **Multimodal RAG chain** - answer questions from retrieved text, tables and images together

## Where this fits
`04_multi_user_conversational_rag_system.ipynb` worked with plain text. This notebook handles a document whose key
information sits in tables and charts.

## Prerequisites
- `02_rag_system_for_question_answering.ipynb` (LCEL RAG chain) - background reading
- Dependencies installed with `uv sync` in this project's folder
- `OPENAI_API_KEY` in this project's `.env` (OpenAI embeddings and `gpt-4o-mini`) - `gpt-4o-mini` also reads the images
- System packages for Unstructured's `hi_res` strategy: Tesseract OCR and Poppler (see Part 1)
- A Redis server on `localhost:6379` for the docstore (see Part 4)
- Internet access: Part 1 downloads the source PDF and NLTK data on the first run

---
## 📦 Step 1: Environment Setup

Load `OPENAI_API_KEY` from this project's `.env`, download the NLTK data Unstructured needs, then create the multimodal chat model.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from this project's .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

### NLTK data for Unstructured

Unstructured uses NLTK tokenizers and taggers. The next cell downloads them with normal TLS certificate verification.

**Security note:** the original version of this cell switched certificate verification off for the whole process, a
workaround for Python installs without root certificates (common with python.org builds on macOS). With verification
off, any server can impersonate the download site. Here that fallback is opt-in: it runs only if you set
`ALLOW_UNVERIFIED_NLTK_DOWNLOAD = True`, only after a verified download failed, and it switches verification back on
right after. Prefer the proper fix: run `Install Certificates.command` from your Python folder.

In [ ]:
# ============================================================================
# NLTK DATA: Tokenizers and taggers used by Unstructured
# ============================================================================
# to prevent 403 errors with unstructured.io till they fix it
# refer: https://github.com/Unstructured-IO/unstructured/issues/3795
import ssl

import nltk

NLTK_PACKAGES = ["punkt", "punkt_tab", "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]

# SECURITY: leave this False. True retries a failed download with TLS certificate verification OFF,
# a workaround for Python installs without root certificates (python.org builds on macOS); the proper
# fix is running "Install Certificates.command" from the Python folder.
ALLOW_UNVERIFIED_NLTK_DOWNLOAD = False


def download_nltk_data():
    return all(nltk.download(package, quiet=True) for package in NLTK_PACKAGES)


try:
    ok = download_nltk_data()
    if not ok and ALLOW_UNVERIFIED_NLTK_DOWNLOAD:
        print("⚠️ Verified NLTK download failed - retrying WITHOUT certificate verification")
        default_https_context = ssl._create_default_https_context
        ssl._create_default_https_context = ssl._create_unverified_context
        try:
            ok = download_nltk_data()
        finally:
            ssl._create_default_https_context = default_https_context  # verification back on
    print("✅ NLTK data downloaded successfully!" if ok else "❌ Some NLTK data could not be downloaded (see the note above)")
except Exception as e:
    print(f"❌ NLTK download error (continuing anyway): {e}")
    print("Note: This may not affect the main functionality")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: gpt-4o-mini (multimodal) for summaries and answers
# ============================================================================
from aied_helpers import get_openai_llm

# temperature=0 is the factory default, as in the original lesson
llm = get_openai_llm(model="gpt-4o-mini")

print(f"✅ LLM initialized: {llm.model_name}")

---
## 📄 Part 1: Partition the PDF into Text, Tables and Images

The source document is a short Congressional Research Service brief on U.S. wildfire statistics, with a table and charts.
The next cell downloads it to `data/IF10244.pdf` if it isn't there yet.

Unstructured's `hi_res` strategy needs two system packages: Tesseract (OCR) and Poppler (PDF rendering). Install them
once, outside the notebook:
- macOS: `brew install tesseract poppler`
- Ubuntu/Debian (and Colab): `sudo apt-get install tesseract-ocr poppler-utils`

In [ ]:
# ============================================================================
# SOURCE PDF: Download once and clear old figures
# ============================================================================
import os
import shutil

import requests

# the U.S. Congressional Research Service brief on wildfire statistics (IF10244)
doc = '../data/IF10244.pdf'
if not os.path.exists(doc):
    response = requests.get('https://sgp.fas.org/crs/misc/IF10244.pdf', timeout=60)
    response.raise_for_status()
    with open(doc, 'wb') as f:
        f.write(response.content)
print('✅ PDF ready:', doc)

# start from an empty ./figures folder so only this run's extracted images are summarized
shutil.rmtree('./figures', ignore_errors=True)

We run the loader function twice to first extract all tables and then load all text and chunk it.

Reason for this is in the latest version of unstructured, you lose the table elements if you chunk directly.
Refer to: https://github.com/Unstructured-IO/unstructured/issues/3827 for more details

So till they push a fix, we will use this method. Once fixed we will update the following loader section.

In [ ]:
# ============================================================================
# PARTITION (pass 1): Extract table elements and images
# ============================================================================
from langchain_community.document_loaders import UnstructuredPDFLoader

# Extract tables
# takes 1-2 min
loader = UnstructuredPDFLoader(file_path=doc,
                               strategy='hi_res',
                               extract_images_in_pdf=True,
                               infer_table_structure=True,
                               mode='elements',
                               image_output_dir_path='./figures')
data = loader.load()

In [ ]:
[doc.metadata['category'] for doc in data if doc.metadata['category'] == 'Table']

In [ ]:
tables = [doc for doc in data if doc.metadata['category'] == 'Table']

In [ ]:
# ============================================================================
# PARTITION (pass 2): Chunk the text by title
# ============================================================================
# Chunk text and extract text content
loader = UnstructuredPDFLoader(file_path=doc,
                               strategy='hi_res',
                               extract_images_in_pdf=True,
                               infer_table_structure=True,
                               chunking_strategy="by_title", # section-based chunking
                               max_characters=4000, # max size of chunks
                               new_after_n_chars=4000, # preferred size of chunks
                               combine_text_under_n_chars=2000, # smaller chunks < 2000 chars will be combined into a larger chunk
                               mode='elements',
                               image_output_dir_path='./figures')
texts = loader.load()

In [ ]:
len(texts)

In [ ]:
data = texts + tables

In [ ]:
len(data)

In [ ]:
data[5]

In [ ]:
from IPython.display import HTML, display, Markdown

In [ ]:
print(data[5].page_content)

In [ ]:
data[5].metadata['text_as_html']

In [ ]:
display(Markdown(data[5].metadata['text_as_html']))

---
## 🧾 Part 2: Prepare Tables and Images

Since unstructured extracts the text from the table without any borders, we can use the HTML text and put it directly in prompts (LLMs understand HTML tables well) or even better convert HTML tables to Markdown tables as below

In [ ]:
import htmltabletomd

md_table = htmltabletomd.convert_table(data[5].metadata['text_as_html'])
print(md_table)

## Separate Data into Text and Table Elements

In [ ]:
# ============================================================================
# SEPARATE: Text chunks (CompositeElement) vs tables (Table)
# ============================================================================
docs = []
tables = []

for doc in data:
    if doc.metadata['category'] == 'Table':
        tables.append(doc)
    elif doc.metadata['category'] == 'CompositeElement':
        docs.append(doc)

len(docs), len(tables)

### Convert HTML Tables to Markdown

In [ ]:
for table in tables:
    table.page_content = htmltabletomd.convert_table(table.metadata['text_as_html'])

In [ ]:
for table in tables:
    print(table.page_content)
    print()

### View Extracted Images

In [ ]:
import os

# the images Unstructured extracted from the PDF
for name in sorted(os.listdir('./figures')):
    print(name, os.path.getsize(os.path.join('./figures', name)), 'bytes')

In [ ]:
from IPython.display import Image

Image('./figures/figure-1-1.jpg')

In [ ]:
Image('./figures/figure-1-2.jpg')

In [ ]:
Image('./figures/figure-1-3.jpg')

---
## ✍️ Part 3: Summarize Text, Tables and Images

We will use the multimodal `llm` (gpt-4o-mini) to produce table and text summaries.

Text summaries are advised if using large chunk sizes (e.g., as set above, we use 4k token chunks).

Summaries are used to retrieve raw tables and / or raw chunks of text.

In [ ]:
# ============================================================================
# TEXT + TABLE SUMMARIES: One batched summary chain over all elements
# ============================================================================
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

# Prompt
prompt_text = """
You are an assistant tasked with summarizing tables and text particularly for semantic retrieval.
These summaries will be embedded and used to retrieve the raw text or table elements
Give a detailed summary of the table or text below that is well optimized for retrieval.
For any tables also add in a one line description of what the table is about besides the summary.
Do not add redundant words like Summary.
Just output the actual summary content.

Table or text chunk:
{element}
"""
prompt = ChatPromptTemplate.from_template(prompt_text)

# Summary chain
summarize_chain = (
                    {"element": RunnablePassthrough()}
                      |
                    prompt
                      |
                    llm
                      |
                    StrOutputParser() # extracts the response as text and returns it as a string
)

# Initialize empty summaries
text_summaries = []
table_summaries = []

text_docs = [doc.page_content for doc in docs]
table_docs = [table.page_content for table in tables]

text_summaries = summarize_chain.batch(text_docs, {"max_concurrency": 5})
table_summaries = summarize_chain.batch(table_docs, {"max_concurrency": 5})

len(text_summaries), len(table_summaries)

In [ ]:
text_summaries[0]

In [ ]:
table_summaries[0]

### Image summaries

We will use the same multimodal `llm` (gpt-4o-mini, from the [GPT-4o](https://openai.com/index/hello-gpt-4o/) family) to produce the image summaries.

* We pass base64 encoded images

In [ ]:
# ============================================================================
# IMAGE SUMMARIES: Send each base64-encoded figure to the multimodal LLM
# ============================================================================
import base64
import os

from langchain_core.messages import HumanMessage


def encode_image(image_path):
    """Getting the base64 string"""
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")


def image_summarize(img_base64, prompt):
    """Make image summary"""
    # the same multimodal chat model as everywhere else in this notebook
    msg = llm.invoke(
        [
            HumanMessage(
                content=[
                    {"type": "text", "text": prompt},
                    {
                        "type": "image_url",
                        "image_url": {"url": f"data:image/jpeg;base64,{img_base64}"},
                    },
                ]
            )
        ]
    )
    return msg.content


def generate_img_summaries(path):
    """
    Generate summaries and base64 encoded strings for images
    path: Path to list of .jpg files extracted by Unstructured
    """

    # Store base64 encoded images
    img_base64_list = []

    # Store image summaries
    image_summaries = []

    # Prompt
    prompt = """You are an assistant tasked with summarizing images for retrieval.
                Remember these images could potentially contain graphs, charts or tables also.
                These summaries will be embedded and used to retrieve the raw image for question answering.
                Give a detailed summary of the image that is well optimized for retrieval.
                Do not add additional words like Summary, This image represents, etc.
             """

    # Apply to images
    for img_file in sorted(os.listdir(path)):
        if img_file.endswith(".jpg"):
            img_path = os.path.join(path, img_file)
            base64_image = encode_image(img_path)
            img_base64_list.append(base64_image)
            image_summaries.append(image_summarize(base64_image, prompt))

    return img_base64_list, image_summaries


# Image summaries
IMG_PATH = './figures'
imgs_base64, image_summaries = generate_img_summaries(IMG_PATH)

In [ ]:
len(imgs_base64), len(image_summaries)

In [ ]:
display(Image('./figures/figure-1-2.jpg'))

In [ ]:
image_summaries[1]

---
## 🗂️ Part 4: Multi-Vector Retriever

Use [multi-vector-retriever](https://python.langchain.com/docs/modules/data_connection/retrievers/multi_vector#summary) to index image (and / or text, table) summaries, but retrieve raw images (along with raw texts or tables).

### Download and Install Redis as a DocStore

You can use any other database or cache as a docstore to store the raw text, table and image elements

The retriever below connects to Redis on `localhost:6379`, so start a Redis server before running it:
- macOS: `brew install redis && brew services start redis`
- Docker (any OS): `docker run -d --name redis -p 6379:6379 redis/redis-stack-server:latest`
- Ubuntu/Debian (and Colab): run the next cell by hand. It is tagged `skip-execution` because it installs a system
  package with `sudo apt-get` and starts a server, which only works on Debian-based Linux.

In [ ]:
%%sh
curl -fsSL https://packages.redis.io/gpg | sudo gpg --dearmor -o /usr/share/keyrings/redis-archive-keyring.gpg
echo "deb [signed-by=/usr/share/keyrings/redis-archive-keyring.gpg] https://packages.redis.io/deb $(lsb_release -cs) main" | sudo tee /etc/apt/sources.list.d/redis.list
sudo apt-get update  > /dev/null 2>&1
sudo apt-get install redis-stack-server  > /dev/null 2>&1
redis-stack-server --daemonize yes

### Open AI Embedding Models

LangChain enables us to access Open AI embedding models which include the newest models: a smaller and highly efficient `text-embedding-3-small` model, and a larger and more powerful `text-embedding-3-large` model.

In [ ]:
# ============================================================================
# EMBEDDINGS: OpenAI text-embedding-3-small
# ============================================================================
from langchain_openai import OpenAIEmbeddings

# details here: https://openai.com/blog/new-embedding-models-and-api-updates
openai_embed_model = OpenAIEmbeddings(model='text-embedding-3-small')

In [ ]:
docs[0]

### Add to vectorstore & docstore

Add raw docs and doc summaries to [Multi Vector Retriever](https://python.langchain.com/docs/modules/data_connection/retrievers/multi_vector#summary):

* Store the raw texts, tables, and images in the `docstore` (here we are using Redis).
* Store the texts, table summaries, and image summaries and their corresponding embeddings in the `vectorstore` (here we are using Chroma) for efficient semantic retrieval.
* Connect them using a common `document_id`

In [ ]:
# ============================================================================
# MULTI-VECTOR RETRIEVER: summaries in Chroma, raw elements in Redis
# ============================================================================
import uuid

from langchain_classic.retrievers.multi_vector import MultiVectorRetriever
from langchain_community.storage import RedisStore
from langchain_community.utilities.redis import get_client
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings


def create_multi_vector_retriever(
    docstore, vectorstore, text_summaries, texts, table_summaries, tables, image_summaries, images
):
    """
    Create retriever that indexes summaries, but returns raw images or texts
    """


    id_key = "doc_id"

    # Create the multi-vector retriever
    retriever = MultiVectorRetriever(
        vectorstore=vectorstore,
        docstore=docstore,
        id_key=id_key,
    )

    # Helper function to add documents to the vectorstore and docstore
    def add_documents(retriever, doc_summaries, doc_contents):
        doc_ids = [str(uuid.uuid4()) for _ in doc_contents]
        summary_docs = [
            Document(page_content=s, metadata={id_key: doc_ids[i]})
            for i, s in enumerate(doc_summaries)
        ]
        retriever.vectorstore.add_documents(summary_docs)
        retriever.docstore.mset(list(zip(doc_ids, doc_contents)))

    # Add texts, tables, and images
    # Check that text_summaries is not empty before adding
    if text_summaries:
        add_documents(retriever, text_summaries, texts)
    # Check that table_summaries is not empty before adding
    if table_summaries:
        add_documents(retriever, table_summaries, tables)
    # Check that image_summaries is not empty before adding
    if image_summaries:
        add_documents(retriever, image_summaries, images)

    return retriever


# The vectorstore to use to index the summaries and their embeddings
chroma_db = Chroma(
    collection_name="mm_rag",
    embedding_function=openai_embed_model,
    collection_metadata={"hnsw:space": "cosine"},
)

# Initialize the storage layer - to store raw images, text and tables
client = get_client('redis://localhost:6379')
redis_store = RedisStore(client=client) # you can use filestore, memorystory, any other DB store also

# Create retriever
retriever_multi_vector = create_multi_vector_retriever(
    redis_store,
    chroma_db,
    text_summaries,
    text_docs,
    table_summaries,
    table_docs,
    image_summaries,
    imgs_base64,
)

In [ ]:
retriever_multi_vector

---
## 🔍 Part 5: Multimodal RAG

### Test the Multimodal Retriever

The retriever searches the summaries and returns the raw elements: text and Markdown tables as bytes, images as base64 strings.

In [ ]:
# ============================================================================
# DISPLAY HELPER: Show a base64-encoded image
# ============================================================================
from IPython.display import HTML, display, Image
from PIL import Image
import base64
from io import BytesIO

def plt_img_base64(img_base64):
    """Disply base64 encoded string as image"""
    # Decode the base64 string
    img_data = base64.b64decode(img_base64)
    # Create a BytesIO object
    img_buffer = BytesIO(img_data)
    # Open the image using PIL
    img = Image.open(img_buffer)
    display(img)

### Check Retrieval

Examine retrieval; we get back images and tables also that are relevant to our question. Which elements come back depends on the summaries, so check the list before displaying `docs[2]` as an image.

In [ ]:
# Check retrieval
query = "Analyze the wildfires trend with acres burned over the years"
docs = retriever_multi_vector.invoke(query)

# number of raw elements returned (at most 4: Chroma's default k)
len(docs)

In [ ]:
docs

In [ ]:
plt_img_base64(docs[2])

In [ ]:
# Check retrieval
query = "Tell me about the percentage of residences burned by wildfires in 2022"
docs = retriever_multi_vector.invoke(query)

# number of raw elements returned (at most 4: Chroma's default k)
len(docs)

In [ ]:
docs

### Utilities to separate retrieved elements

We need to bin the retrieved doc(s) into the correct parts of the GPT-4o prompt template.

Here we need to have text, table elements as one set of inputs and image elements as the other set of inputs as both require separate prompts in GPT-4o.

In [ ]:
# ============================================================================
# SPLIT HELPERS: Tell base64 images apart from text and tables
# ============================================================================
import re
import base64

def looks_like_base64(sb):
    """Check if the string looks like base64"""
    return re.match("^[A-Za-z0-9+/]+[=]{0,2}$", sb) is not None


def is_image_data(b64data):
    """
    Check if the base64 data is an image by looking at the start of the data
    """
    image_signatures = {
        b"\xff\xd8\xff": "jpg",
        b"\x89\x50\x4e\x47\x0d\x0a\x1a\x0a": "png",
        b"\x47\x49\x46\x38": "gif",
        b"\x52\x49\x46\x46": "webp",
    }
    try:
        header = base64.b64decode(b64data)[:8]  # Decode and get the first 8 bytes
        for sig, format in image_signatures.items():
            if header.startswith(sig):
                return True
        return False
    except Exception:
        return False


def split_image_text_types(docs):
    """
    Split base64-encoded images and texts
    """
    b64_images = []
    texts = []
    for doc in docs:
        # Check if the document is of type Document and extract page_content if so
        if isinstance(doc, Document):
            doc = doc.page_content.decode('utf-8')
        else:
            doc = doc.decode('utf-8')
        if looks_like_base64(doc) and is_image_data(doc):
            b64_images.append(doc)
        else:
            texts.append(doc)
    return {"images": b64_images, "texts": texts}

In [ ]:
# Check retrieval
query = "Tell me detailed statistics of the top 5 years with largest wildfire acres burned"
docs = retriever_multi_vector.invoke(query)

# number of raw elements returned (at most 4: Chroma's default k)
len(docs)

In [ ]:
docs

In [ ]:
is_image_data(docs[2].decode('utf-8'))

In [ ]:
r = split_image_text_types(docs)
r

In [ ]:
plt_img_base64(r['images'][0])

### Build End-to-End Multimodal RAG Pipeline

Now let's connect our retriever, prompt instructions and build a multimodal RAG chain

In [ ]:
# ============================================================================
# MULTIMODAL RAG CHAIN: retrieve -> split images/texts -> multimodal prompt -> LLM
# ============================================================================
from operator import itemgetter
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.messages import HumanMessage

def multimodal_prompt_function(data_dict):
    """
    Create a multimodal prompt with both text and image context.

    This function formats the provided context from `data_dict`, which contains
    text, tables, and base64-encoded images. It joins the text (with table) portions
    and prepares the image(s) in a base64-encoded format to be included in a message.

    The formatted text and images (context) along with the user question are used to
    construct a prompt for GPT-4o
    """
    formatted_texts = "\n".join(data_dict["context"]["texts"])
    messages = []

    # Adding image(s) to the messages if present
    if data_dict["context"]["images"]:
        for image in data_dict["context"]["images"]:
            image_message = {
                "type": "image_url",
                "image_url": {"url": f"data:image/jpeg;base64,{image}"},
            }
            messages.append(image_message)

    # Adding the text and tables for analysis
    text_message = {
        "type": "text",
        "text": (
            f"""You are an analyst tasked with understanding detailed information and trends
                from text documents, data tables, and charts and graphs in images.
                You will be given context information below which will be a mix of text, tables,
                and images usually of charts or graphs.
                Use this information to provide answers related to the user question.
                Analyze all the context information including tables, text and images to generate the answer.
                Do not make up answers, use the provided context documents below
                and answer the question to the best of your ability.

                User question:
                {data_dict['question']}

                Context documents:
                {formatted_texts}

                Answer:
            """
        ),
    }
    messages.append(text_message)
    return [HumanMessage(content=messages)]


# Create RAG chain
multimodal_rag = (
        {
            "context": itemgetter('context'),
            "question": itemgetter('input'),
        }
            |
        RunnableLambda(multimodal_prompt_function)
            |
        llm
            |
        StrOutputParser()
)

# Pass input query to retriever and get context document elements
retrieve_docs = (itemgetter('input')
                    |
                retriever_multi_vector
                    |
                RunnableLambda(split_image_text_types))

# Below, we chain `.assign` calls. This takes a dict and successively
# adds keys-- "context" and "answer"-- where the value for each key
# is determined by a Runnable (function or chain executing at runtime).
# This helps in also having the retrieved context along with the answer generated by GPT-4o
multimodal_rag_w_sources = (RunnablePassthrough.assign(context=retrieve_docs)
                                               .assign(answer=multimodal_rag)
)

In [ ]:
# Run RAG chain
query = "Tell me detailed statistics of the top 5 years with largest wildfire acres burned"
response = multimodal_rag_w_sources.invoke({'input': query})
response

In [ ]:
# ============================================================================
# QA HELPER: Print the answer, then every text and image source
# ============================================================================
def multimodal_rag_qa(query):
    response = multimodal_rag_w_sources.invoke({'input': query})
    print('=='*50)
    print('Answer:')
    display(Markdown(response['answer']))
    print('--'*50)
    print('Sources:')
    text_sources = response['context']['texts']
    img_sources = response['context']['images']
    for text in text_sources:
        display(Markdown(text))
        print()
    for img in img_sources:
        plt_img_base64(img)
        print()
    print('=='*50)

In [ ]:
query = "Tell me detailed statistics of the top 5 years with largest wildfire acres burned"
multimodal_rag_qa(query)

In [ ]:
# Run RAG chain
query = "Analyze the wildfires trend with acres burned over the years"
multimodal_rag_qa(query)

In [ ]:
# Run RAG chain
query = "Tell me about the percentage of residences burned by wildfires in 2022"
multimodal_rag_qa(query)

---
## 📝 Summary

In this notebook, we learned:

### 1. Partition a PDF
- `UnstructuredPDFLoader` with `strategy='hi_res'` extracts text, tables (with HTML) and images from a PDF.
- Two passes keep the table elements that by-title chunking would otherwise drop.

### 2. Prepare Tables and Images
- HTML tables convert cleanly to Markdown, which LLMs read well; extracted figures are plain JPG files.

### 3. Summarize Every Element
- A multimodal LLM writes retrieval-optimized summaries of text, tables and images (images sent as base64).

### 4. Multi-Vector Retriever
- Summaries are embedded in Chroma; the raw elements live in a docstore (Redis), linked by a shared `doc_id`.

### 5. Multimodal RAG Chain
- Retrieved elements are split into images and texts, and both go into one multimodal prompt.

### Next Steps
- This is the last notebook of the project. The project's `README.md` lists all five builds; swap Redis for any other
  LangChain byte store, or point the loader at your own PDFs.